# Creational Design Patterns

In [13]:
from abc import ABC, abstractmethod

## Exercise 1: Payment System using the Factory Pattern

### 1) Abstract Base Class

In [14]:
class PaymentProcessor(ABC):

    @abstractmethod
    def validate(self, info: dict) -> bool:
        pass

    @abstractmethod
    def process(self, total: float, info: dict) -> dict:
        pass


### 2) Concrete Classes

In [15]:
class CreditCardProcessor(PaymentProcessor):
    def validate(self, info):
        card_ok = info.get("card_number") and len(info["card_number"]) == 16
        cvv_ok = info.get("cvv") and len(info["cvv"]) == 3
        return card_ok and cvv_ok

    def process(self, total, info):
        if not self.validate(info):
            return {"success": False, "error": "Invalid credit card details"}

        transaction_fee = total * 0.029
        return {
            "success": True,
            "method": "credit_card",
            "amount": total + transaction_fee,
            "fee": transaction_fee
        }


class BankTransferProcessor(PaymentProcessor):
    def validate(self, info):
        return info.get("iban") and len(info["iban"]) >= 15

    def process(self, total, info):
        if not self.validate(info):
            return {"success": False, "error": "Invalid IBAN provided"}

        flat_fee = 1.50
        return {
            "success": True,
            "method": "bank_transfer",
            "amount": total + flat_fee,
            "fee": flat_fee
        }


class PayPalProcessor(PaymentProcessor):
    def validate(self, info):
        return info.get("email") and "@" in info["email"]

    def process(self, total, info):
        if not self.validate(info):
            return {"success": False, "error": "Invalid PayPal email address"}

        service_fee = total * 0.034 + 0.30
        return {
            "success": True,
            "method": "paypal",
            "amount": total + service_fee,
            "fee": service_fee
        }

### 3) Factory Class

In [16]:
class PaymentFactory:
    _registry = {
        "credit_card": CreditCardProcessor,
        "bank_transfer": BankTransferProcessor,
        "paypal": PayPalProcessor
    }

    def get_processor(self, payment_type: str) -> PaymentProcessor:
        processor_cls = self._registry.get(payment_type)
        if processor_cls is None:
            raise ValueError(f"Unsupported payment type: '{payment_type}'")
        return processor_cls()

### Example Usage

In [17]:
factory = PaymentFactory()
processor = factory.get_processor("credit_card")
result = processor.process(100, {"card_number": "1234567890123456", "cvv": "456"})
print(result)

{'success': True, 'method': 'credit_card', 'amount': 102.9, 'fee': 2.9000000000000004}


## Exercise 2: Employee Onboarding with the Builder Pattern

In [18]:
from dataclasses import dataclass

### 1) Data Model

In [19]:
@dataclass
class Employee:
    first_name: str,
    last_name: str,
    email: str,
    department: str,
    position: str,
    salary: float,
    start_date: str,
    manager_id: int = None,
    phone: str = None,
    address: str = None,
    emergency_contact: str = None,
    has_parking: bool = False,
    has_laptop: bool = False,
    has_vpn_access: bool = False,
    has_admin_rights: bool = False,
    office_location: str = None,
    contract_type: str = "permanent"

### 2) Builder

In [20]:
class EmployeeBuilder:

    def __init__(self):
        self.employee = Employee("", "", "")

    def with_name(self, first_name, last_name):
        self.employee.first_name = first_name
        self.employee.last_name = last_name
        return self

    def with_email(self, email):
        if "@" not in email:
            raise ValueError("Email address is not valid")
        self.employee.email = email
        return self

    def with_job(self, dept, position, salary):
        self.employee.department = dept
        self.employee.position = position
        self.employee.salary = salary
        return self

    def with_equipment(self, laptop=False, parking=False):
        self.employee.has_laptop = laptop
        self.employee.has_parking = parking
        return self

    def with_access(self, vpn=False, admin=False):
        self.employee.has_vpn_access = vpn
        self.employee.has_admin_rights = admin
        return self

    def build(self):
        if not self.employee.first_name or not self.employee.last_name:
            raise ValueError("Employee must have a first and last name")
        return self.employee

In [24]:
# Fluent builder pattern — readable step-by-step construction
employee = (
    EmployeeBuilder()
    .with_name("Alice", "Martin")
    .with_email("alice.martin@company.com")
    .with_job("Data Science", "ML Engineer", 82000)
    .with_equipment(laptop=True, parking=True)
    .with_access(vpn=True, admin=False)
    .build()
)

## Exercise 3: Singleton Configuration Manager

In [26]:
import json

In [30]:
class ConfigSource(ABC):
    @abstractmethod
    def load(self):
        pass


class JsonFileConfigSource(ConfigSource):
    def __init__(self, filepath="config.json"):
        self._filepath = filepath

    def load(self):
        with open(self._filepath, "r", encoding="utf-8") as f:
            return json.load(f)


class ConfigManager:
    _instance = None

    def __init__(self, source):
        self._config = source.load()

    @classmethod
    def get_instance(cls, source=None):
        if cls._instance is None:
            if source is None:
                source = JsonFileConfigSource("config.json")
            cls._instance = cls(source)
        return cls._instance

    def get(self, key, default=None):
        node = self._config
        for part in key.split("."):
            if isinstance(node, dict) and part in node:
                node = node[part]
            else:
                return default
        return node
